# Suite LOG — The logical layout

`Logical` defines a diagram's logical layout: diagrams, shapes and points, each owned, vectors and rotations, with no coordinates, and resolves the compass of a point or a shape: mbse-basis [CAP-237: logical layout schemas](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-237.md), by the rules of [the diagrams proposal](https://github.com/pitaman71/mbse-basis/blob/main/proposals/DIAGRAMS.md).

In [ ]:
from mbse.Schemas.Framework import JSON, Builders, Paths, Validators
from mbse.Diagrams.Framework import Logical

NAMES = ["Logical.OfRotation", "Logical.InDiagrams", "Logical.DiagramDirectory", "Logical.OfDiagram",
         "Logical.InDiagram", "Logical.OfShape", "Logical.Containment", "Logical.Adjacency", "Logical.OfPoint",
         "Logical.Origin", "Logical.Borders", "Logical.Extents", "Logical.Vector"]
store = Builders.Store()
store.transaction(Logical.install)
L = store.Logical
diagrams = store.singleton("Logical.Diagrams")
problems = Validators.Validate(store).Reachable


def path(x):
    """An object's path, as JSON text."""
    return JSON.ToJSON(store)(store.Paths.OfRoot.as_schema(), Paths.of(x))


def turn(degrees):
    return lambda r: r.degrees(degrees)


def origin(s):
    return s.origin(lambda a: a.insert(lambda e: e.point(lambda p: p)))


def named(name):
    return lambda a: a.insert(lambda e: e.name(name).point(lambda p: p))


def vector(start, end, degrees):
    L.OfPoint(start, lambda p: p.outgoing(lambda a: a.insert(lambda e: e.end(end).rotation(turn(degrees)))))


def raises(message, block):
    try:
        with store.transaction():
            block()
    except ValueError as error:
        assert str(error) == message, str(error)
    else:
        raise AssertionError(f"expected ValueError: {message}")


with store.transaction():
    L.DiagramDirectory(diagrams, lambda f: f.diagrams(lambda a: a.insert(lambda e: e.name("overview").diagram(
        lambda d: d.shapes(lambda s: s.insert(lambda e: e.name("cpu").shape(lambda c: origin(c)
            .borders(named("ne")).extents(named("e"))
            .children(lambda k: k.insert(lambda e: e.name("alu").child(lambda x: origin(x).borders(named("n"))))))))))))
overview = diagrams.diagrams({"name": "overview"})
cpu = overview.shapes({"name": "cpu"})
alu = cpu.children({"name": "alu"})
(o,) = [e.point for e in cpu.origin]
ne, east = cpu.borders({"name": "ne"}), cpu.extents({"name": "e"})

<a id="LOG-01"></a>
## LOG-01 · The install defines the logical layout's schemas, each valid, and run again writes nothing

In [ ]:
schemas = [store.registered(name) for name in NAMES]
assert [schema.validate() for schema in schemas] == [[]] * len(NAMES)
store.transaction(Logical.install)
assert [store.registered(name) for name in NAMES] == schemas
assert diagrams is store.Instances.Logical.Diagrams

<a id="LOG-02"></a>
## LOG-02 · Diagrams, shapes and points are owned: each found by its key, its path read back to Logical.Diagrams

In [ ]:
assert [e.diagram for e in diagrams.diagrams] == [overview] and [e.shape for e in overview.shapes] == [cpu]
assert [e.child for e in cpu.children] == [alu] and [e.container for e in alu.container] == [cpu]
assert [e.shape for e in o.origin_of] == [cpu] and [e.shape for e in ne.border_of] == [cpu]
assert [e.shape for e in east.extent_of] == [cpu]
assert path(alu.borders({"name": "n"})) == (
    '{"singleton": "Logical.Diagrams", "next": {"entry": {"i": "diagrams", "o": "directory", "properties"'
    ': {"name": "overview"}, "next": {"entry": {"i": "shapes", "o": "diagram", "properties": {"name": "cp'
    'u"}, "next": {"entry": {"i": "children", "o": "container", "properties": {"name": "alu"}, "next": {"'
    'entry": {"i": "borders", "o": "border_of", "properties": {"name": "n"}, "next": {"target": {}}}}}}}}'
    '}}}')
assert problems(L.OfDiagram, overview) == []
raises("a 'Logical.OfPoint' needs an owner through 'origin_of', 'border_of', 'extent_of': give it one, since no "
       "singleton can be", lambda: L.OfPoint(lambda p: p))
raises("a 'Logical.OfShape' needs an owner through 'diagram', 'container': give it one, since no singleton can be",
       lambda: L.OfShape(lambda s: s))

<a id="LOG-03"></a>
## LOG-03 · A shape has one origin, and a point one owner, as the validator reports

In [ ]:
with store.transaction():
    L.OfShape(cpu, origin)
assert problems(L.OfShape, cpu) == ["unique(point) violated: entries agreeing on ['shape'] differ on ['point']"]
(extra,) = [e.point for e in cpu.origin if e.point is not o]
with store.transaction():
    L.OfShape(cpu, lambda s: s.origin(lambda a: a.remove(lambda e: e.point(extra))))
assert [e.point for e in cpu.origin] == [o] and problems(L.OfShape, cpu) == []
with store.transaction():
    L.OfShape(alu, lambda s: s.borders(lambda a: a.insert(lambda e: e.name("ne").point(ne))))
assert problems(L.OfShape, alu) == [
    "unique(name, shape) violated: entries agreeing on ['point'] differ on ['name', 'shape']"]
with store.transaction():
    L.OfShape(alu, lambda s: s.borders(lambda a: a.remove(lambda e: e.name("ne").point(ne))))
assert problems(L.OfDiagram, overview) == []

<a id="LOG-04"></a>
## LOG-04 · A vector has one rotation per start and end

In [ ]:
with store.transaction():
    vector(o, ne, 45.0)
(v,) = ne.incoming
assert v.start is o and v.end is ne and v.rotation.degrees.get() == 45.0 and problems(L.OfShape, cpu) == []
with store.transaction():
    vector(o, ne, 135.0)
assert problems(L.OfShape, cpu) == [
    "unique(rotation) violated: entries agreeing on ['end', 'start'] differ on ['rotation']"]
with store.transaction():
    L.OfPoint(o, lambda p: p.outgoing(lambda a: a.remove(lambda e: e.end(ne).rotation(turn(135.0)))))
assert [e.rotation.degrees.get() for e in ne.incoming] == [45.0] and problems(L.OfShape, cpu) == []

<a id="LOG-05"></a>
## LOG-05 · A container's children, and adjacencies between siblings by rotation

In [ ]:
with store.transaction():
    L.OfShape(cpu, lambda s: s.children(lambda k: k.insert(lambda e: e.name("fpu").child(origin))))
fpu = cpu.children({"name": "fpu"})
with store.transaction():
    L.OfShape(alu, lambda s: s.adjacent_to(lambda a: a.insert(lambda e: e.to_shape(fpu).rotation(turn(90.0)))))
assert [e.child for e in cpu.children] == [alu, fpu]
(adjacency,) = fpu.adjacent_from
assert adjacency.from_shape is alu and adjacency.rotation.degrees.get() == 90.0 and problems(L.OfShape, cpu) == []

<a id="LOG-06"></a>
## LOG-06 · A compass: a shape's and its origin's north, a border or extent point's its vector's rotation

In [ ]:
n = alu.borders({"name": "n"})
with store.transaction():
    L.OfShape(cpu, lambda s: s.extents(named("sw")))
    vector(o, east, -90.0)
    vector(o, cpu.extents({"name": "sw"}), 585.5)
    vector([e.point for e in alu.origin][0], n, 0.0)
sw = cpu.extents({"name": "sw"})
assert (Logical.compass(cpu), Logical.compass(o), Logical.compass(alu)) == (0.0, 0.0, 0.0)
assert (Logical.compass(ne), Logical.compass(east), Logical.compass(sw), Logical.compass(n)) == (45.0, 270.0, 225.5, 0.0)

<a id="LOG-07"></a>
## LOG-07 · A compass the rules leave undefined is refused

In [ ]:
with store.transaction():
    L.OfShape(cpu, lambda s: s.borders(named("w")).borders(named("s")).children(lambda k: k.insert(
        lambda e: e.name("bare").child(lambda x: x.borders(named("x"))))))
    L.OfShape(fpu, lambda s: s.borders(named("q")))
    w, south, bare = cpu.borders({"name": "w"}), cpu.borders({"name": "s"}), cpu.children({"name": "bare"})
    L.OfPoint(o, lambda p: p.outgoing(lambda a: a.insert(lambda e: e.end(south))))
    vector(o, w, 1.0)
    vector(o, w, 2.0)
raises("a shape without an origin has no compass", lambda: Logical.compass(bare))
raises("a shape without an origin has no compass", lambda: Logical.compass(bare.borders({"name": "x"})))
raises("a point with 0 vectors from its shape's origin, not one, has no compass",
       lambda: Logical.compass(fpu.borders({"name": "q"})))
raises("a point with 2 vectors from its shape's origin, not one, has no compass", lambda: Logical.compass(w))
raises("a point whose vector from its shape's origin has no rotation has no compass",
       lambda: Logical.compass(south))